# 03. Financial & Time-Series Feature Engineering
การสร้างฟีเจอร์ทางการเงิน (Intraday, Yield Slope, Relative Strength) และฟีเจอร์อนุกรมเวลา (Lags, Momentum, Rolling Volatility, World Indices, Currencies)

### โหลดข้อมูล Intermediate ที่คลีนแล้วจาก data/02_intermediate/
โหลดข้อมูลหุ้นและปัจจัยมหภาคทั้งหมดที่ผ่านการทำความสะอาดแล้วเข้ามาในระบบ เพื่อพร้อมสำหรับการสกัดและสร้างฟีเจอร์ทางการเงิน

In [ ]:
# ==============================================================================
# โหลดข้อมูลคลีนจาก data/02_intermediate/ เข้าสู่หน่วยความจำอัตโนมัติ
# ==============================================================================
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

intermediate_candidates = [
    Path('../data/02_intermediate'),
    Path('data/02_intermediate'),
    Path('./data/02_intermediate')
]
inter_dir = None
for c in intermediate_candidates:
    if c.exists():
        inter_dir = c.resolve()
        break

if inter_dir:
    for f in sorted(inter_dir.glob('df_*.csv')):
        globals()[f.stem] = pd.read_csv(f)
    print(f"✓ โหลดชุดข้อมูลคลีนจาก {inter_dir} สำเร็จแล้ว")
else:
    print("⚠ ไม่พบโฟลเดอร์ data/02_intermediate กรุณารัน 01_data_cleaning.ipynb ก่อน")


## feature engineering

### สูตร feature engineering

In [ ]:
import pandas as pd
import numpy as np

# ฟังก์ชันเสริมสำหรับแปลงคอลัมน์ข้อมูลต่าง ๆ ให้เป็นตัวเลขอย่างปลอดภัย
def safe_to_numeric(series):
    if series.dtype == 'object':
        return pd.to_numeric(series.astype(str).str.replace(',', '').str.rstrip('%'), errors='coerce')
    return pd.to_numeric(series, errors='coerce')

# =====================================================================
# 1. กลุ่มฟังก์ชันคำนวณภายในหลักทรัพย์ (Intraday & Lagged Features)
# =====================================================================

def calculate_intraday_features(df):
    """
    คำนวณหาความผันผวนระหว่างวัน (Volatility) และทิศทางความเคลื่อนไหวราคาภายในวัน (Movement)
    สูตร:
      - Daily_Volatility = High - Low
      - Daily_Movement = Close - Open
    """
    df_out = df.copy()
    df_out['Date'] = pd.to_datetime(df_out['Date'])

    # แปลงราคารองรับกรณีเป็น String
    high = safe_to_numeric(df_out['High'])
    low = safe_to_numeric(df_out['Low'])
    close = safe_to_numeric(df_out['Close'])
    open_val = safe_to_numeric(df_out['Open'])

    df_out['Daily_Volatility'] = high - low
    df_out['Daily_Movement'] = close - open_val
    return df_out

def calculate_price_change_prev_day(df, stock_name=None, close_col='Close'):
    """
    คำนวณส่วนต่างราคาปิดของวันนี้เทียบกับราคาปิดของวันทำการก่อนหน้า
    สูตร: Price_Change_Prev_Day = Close_today - Close_yesterday
    """
    df_out = df.copy()
    df_out['Date'] = pd.to_datetime(df_out['Date'])
    df_out = df_out.sort_values(by='Date')

    # ดึงค่า Close เป็นตัวเลข
    close_series = safe_to_numeric(df_out[close_col])

    # คำนวณหาผลต่างย้อนหลัง 1 วัน
    df_out['Price_Change_Prev_Day'] = close_series.diff()

    if stock_name:
        df_out['Stock'] = stock_name

    return df_out.reset_index(drop=True)

def calculate_volatility_to_open(df):
    """
    คำนวณอัตราส่วนความผันผวนระหว่างวันเทียบราคาเปิด
    สูตร: Volatility_to_Open = (High - Low) / Open
    """
    df_out = df.copy()
    high = safe_to_numeric(df_out['High'])
    low = safe_to_numeric(df_out['Low'])
    open_val = safe_to_numeric(df_out['Open'])

    df_out['Volatility_to_Open'] = (high - low) / open_val
    return df_out

# =====================================================================
# 2. กลุ่มฟังก์ชันเปรียบเทียบข้ามหลักทรัพย์ และ ปัจจัยภายนอก (Cross-File Features)
# =====================================================================

def calculate_relative_return(df_stock, df_factor, factor_name='FX', stock_name=None):
    """
    คำนวณผลตอบแทนส่วนต่างเทียบกับปัจจัยภายนอก (เช่น อัตราแลกเปลี่ยน หรือดอกเบี้ย)
    สูตร: Relative_Return = Pct_Change_Stock - Pct_Change_Factor
    """
    df_stk_cp = df_stock.copy()
    df_fac_cp = df_factor.copy()

    df_stk_cp['Date'] = pd.to_datetime(df_stk_cp['Date'])
    df_fac_cp['Date'] = pd.to_datetime(df_fac_cp['Date'])

    # แปลงเปอร์เซ็นต์การเปลี่ยนแปลงเป็นตัวเลข
    df_stk_cp['Stock_Pct_Change'] = safe_to_numeric(df_stk_cp['Pct_Change'])
    df_fac_cp['Factor_Pct_Change'] = safe_to_numeric(df_fac_cp['Pct_Change'])

    # รวมข้อมูล
    merged = pd.merge(
        df_stk_cp[['Date', 'Stock_Pct_Change', 'Close', 'Open', 'High', 'Low']],
        df_fac_cp[['Date', 'Factor_Pct_Change']].rename(columns={'Factor_Pct_Change': f'{factor_name}_Pct_Change'}),
        on='Date',
        how='inner'
    )

    merged[f'Relative_Return_vs_{factor_name}'] = merged['Stock_Pct_Change'] - merged[f'{factor_name}_Pct_Change']

    if stock_name:
        merged['Stock'] = stock_name

    return merged.sort_values(by='Date').reset_index(drop=True)

def calculate_yield_slope(df_long, df_short, long_name='Yield_20Y', short_name='Yield_10Y'):
    """
    คำนวณความชันระหว่างอัตราผลตอบแทนพันธบัตรรัฐบาลระยะยาวและระยะสั้น
    สูตร: Yield_Curve_Slope = Yield_Long - Yield_Short
    """
    df_l = df_long.copy()
    df_s = df_short.copy()

    df_l['Date'] = pd.to_datetime(df_l['Date'])
    df_s['Date'] = pd.to_datetime(df_s['Date'])

    # แปลง Yield เป็นตัวเลข
    df_l['Yield_Long'] = safe_to_numeric(df_l['Close'])
    df_s['Yield_Short'] = safe_to_numeric(df_s['Close'])

    merged = pd.merge(
        df_l[['Date', 'Yield_Long']].rename(columns={'Yield_Long': long_name}),
        df_s[['Date', 'Yield_Short']].rename(columns={'Yield_Short': short_name}),
        on='Date',
        how='inner'
    )

    merged['Yield_Curve_Slope'] = merged[long_name] - merged[short_name]
    return merged.sort_values(by='Date').reset_index(drop=True)

def calculate_relative_strength(df_export, df_import, exp_stock_name="Export_Stock", imp_stock_name="Import_Stock"):
    """
    คำนวณอัตราส่วนความแข็งแกร่งเปรียบเทียบเชิงสัมพัทธ์ และส่วนต่างผลตอบแทนของคู่หุ้น
    สูตร:
      - Relative_Strength_Ratio = Export_Close / Import_Close
      - Return_Diff_Export_vs_Import = Export_Pct_Change - Import_Pct_Change
    """
    df_exp_sub = df_export.copy()
    df_imp_sub = df_import.copy()

    df_exp_sub['Date'] = pd.to_datetime(df_exp_sub['Date'])
    df_imp_sub['Date'] = pd.to_datetime(df_imp_sub['Date'])

    # แปลง Close และ Pct_Change เป็นตัวเลข
    df_exp_sub['Export_Close'] = safe_to_numeric(df_exp_sub['Close'])
    df_exp_sub['Export_Pct_Change'] = safe_to_numeric(df_exp_sub['Pct_Change'])

    df_imp_sub['Import_Close'] = safe_to_numeric(df_imp_sub['Close'])
    df_imp_sub['Import_Pct_Change'] = safe_to_numeric(df_imp_sub['Pct_Change'])

    merged = pd.merge(
        df_exp_sub[['Date', 'Export_Close', 'Export_Pct_Change']],
        df_imp_sub[['Date', 'Import_Close', 'Import_Pct_Change']],
        on='Date',
        how='inner'
    )

    merged['Stock_Pair'] = f"{exp_stock_name} vs {imp_stock_name}"
    merged['Relative_Strength_Ratio'] = merged['Export_Close'] / merged['Import_Close']
    merged['Return_Diff_Export_vs_Import'] = merged['Export_Pct_Change'] - merged['Import_Pct_Change']

    return merged.sort_values(by='Date').reset_index(drop=True)

def convert_price_to_usd(df_stock, df_fx, stock_name=None):
    """
    แปลงราคาหุ้นปิดสกุลเงินบาท (THB) ให้เป็นสกุลเงินดอลลาร์สหรัฐ (USD) อ้างอิงอัตราแลกเปลี่ยนรายวัน
    สูตร: Stock_Close_USD = Stock_Close_THB / USD_THB_Close
    """
    df_stk_cp = df_stock.copy()
    df_fx_cp = df_fx.copy()

    df_stk_cp['Date'] = pd.to_datetime(df_stk_cp['Date'])
    df_fx_cp['Date'] = pd.to_datetime(df_fx_cp['Date'])

    # แปลงราคาปิดเป็นตัวเลข
    df_stk_cp['Stock_Close_THB'] = safe_to_numeric(df_stk_cp['Close'])
    df_fx_cp['USD_THB_Close'] = safe_to_numeric(df_fx_cp['Close'])

    merged = pd.merge(
        df_stk_cp,
        df_fx_cp[['Date', 'USD_THB_Close']],
        on='Date',
        how='inner'
    )

    merged['Stock_Close_USD'] = merged['Stock_Close_THB'] / merged['USD_THB_Close']

    if stock_name:
        merged['Stock'] = stock_name

    return merged.sort_values(by='Date').reset_index(drop=True)

### ฟังก์ชั่นพลอตกราฟจาก feature engineering

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display

def plot_engineered_feature_scatter(
    stocks_dict,
    feature_formula,
    feature_col,
    feature_label_th,
    title,
    color='#2ca02c'
):
    """
    ฟังก์ชันที่มีความยืดหยุ่นสูงสำหรับพล็อตกราฟ Scatter และหาสหสัมพันธ์ (Correlation)

    Parameters:
    -----------
    stocks_dict : dict
        Dictionary ของข้อมูลหุ้น เช่น {'PTTGC': df_PTTGC, ...}
    feature_formula : function
        ฟังก์ชันคำนวณทาง Feature Engineering เช่น calculate_intraday_features
    feature_col : str
        ชื่อคอลัมน์ของฟีเจอร์ที่เราต้องการวิเคราะห์ เช่น 'Daily_Volatility' หรือ 'Volatility_to_Open'
    feature_label_th : str
        ชื่อภาษาไทยของฟีเจอร์นั้นเพื่อใช้แสดงบนแกนกราฟ
    title : str
        หัวข้อหลักของกราฟทั้งหมด
    color : str
        รหัสสีของจุด Scatter (Default: สีเขียว '#2ca02c')
    """
    # 1. ตั้งค่าฟอนต์ภาษาไทย
    thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
    sns.set_theme(style="whitegrid")
    plt.rcParams['font.family'] = 'Sarabun'
    plt.rcParams['axes.unicode_minus'] = False

    # 2. คำนวณจำนวนแถวสำหรับแสดงผล Grid อัตโนมัติ
    stock_names = list(stocks_dict.keys())
    num_stocks = len(stock_names)
    num_cols = 2
    num_rows = int(np.ceil(num_stocks / num_cols))

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 4.8 * num_rows))
    axes = axes.flatten() if num_stocks > 1 else np.array([axes])

    summary_list = []

    # 3. วนลูปคำนวณและพล็อตกราฟทีละหุ้น
    for idx, name in enumerate(stock_names):
        # ใช้สูตรคำนวณยืดหยุ่นที่ส่งเข้ามาทางพารามิเตอร์
        df_feat = feature_formula(stocks_dict[name])

        # จัดการค่าว่างและดึงเฉพาะข้อมูลที่พร้อมวิเคราะห์
        df_feat = df_feat.dropna(subset=[feature_col, 'Close'])

        vol_series = pd.to_numeric(df_feat[feature_col], errors='coerce')
        close_series = pd.to_numeric(df_feat['Close'], errors='coerce')

        # คำนวณค่าสหสัมพันธ์ (Correlation)
        correlation = vol_series.corr(close_series)

        summary_list.append({
            'หุ้น': name,
            f'ค่าเฉลี่ย {feature_label_th}': vol_series.mean(),
            'Correlation (vs Close)': correlation
        })

        # พล็อตกราฟความสัมพันธ์พร้อม Regression Line
        sns.regplot(
            data=df_feat,
            x='Close',
            y=feature_col,
            ax=axes[idx],
            scatter_kws={'alpha': 0.25, 'color': color, 's': 10},
            line_kws={'color': '#d62728', 'linewidth': 1.5}
        )

        # กำหนดข้อความต่าง ๆ บนกราฟด้วยฟอนต์ไทย
        axes[idx].set_title(f'{name} (Corr = {correlation:.3f})', fontproperties=thai_font, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('ราคาปิด (Close Price)', fontproperties=thai_font, fontsize=11)
        axes[idx].set_ylabel(feature_label_th, fontproperties=thai_font, fontsize=11)

    # ซ่อนส่วนแกนย่อยที่เหลือ (ในกรณีที่จำนวนกลุ่มย่อยเป็นเลขคี่)
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title, fontproperties=thai_font, fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # 4. แสดงตารางผลลัพธ์
    df_summary = pd.DataFrame(summary_list)
    print(f"\n=== [ตารางสรุปผล] ความสัมพันธ์ระหว่าง ราคาปิด และ {feature_label_th} ===")
    display(df_summary)

### calculate_intraday_features

In [ ]:
# จัดกลุ่มข้อมูลสำหรับส่งเข้าฟังก์ชัน
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# --- 1. วิเคราะห์ความผันผวนระหว่างวัน (Daily Volatility) กลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=calculate_intraday_features,
    feature_col='Daily_Volatility',
    feature_label_th='ความผันผวนระหว่างวัน (Daily Volatility)',
    title='การวิเคราะห์ความผันผวนระหว่างวัน (Daily Volatility) ของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 2. วิเคราะห์ความผันผวนระหว่างวัน (Daily Volatility) กลุ่มหุ้นส่งออก---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=calculate_intraday_features,
    feature_col='Daily_Volatility',
    feature_label_th='ความผันผวนระหว่างวัน (Daily Volatility)',
    title='การวิเคราะห์ความผันผวนระหว่างวัน (Daily Volatility) ของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

### calculate_price_change_prev_day (ส่วนต่างราคาปิดเทียบกับวันก่อนหน้า)

In [ ]:
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# --- 1. วิเคราะห์ผลต่างราคาปิดจากวันก่อนหน้า กลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=lambda df: calculate_price_change_prev_day(df),
    feature_col='Price_Change_Prev_Day',
    feature_label_th='ผลต่างราคาปิดจากวันทำการก่อนหน้า (Price Change vs Prev Day)',
    title='การวิเคราะห์ผลต่างราคาปิดจากวันก่อนหน้า ของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 2. วิเคราะห์ผลต่างราคาปิดจากวันก่อนหน้า กลุ่มหุ้นส่งออก ---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=lambda df: calculate_price_change_prev_day(df),
    feature_col='Price_Change_Prev_Day',
    feature_label_th='ผลต่างราคาปิดจากวันทำการก่อนหน้า (Price Change vs Prev Day)',
    title='การวิเคราะห์ผลต่างราคาปิดจากวันก่อนหน้า ของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

### calculate_volatility_to_open (อัตราส่วนความผันผวนระหว่างวันเทียบราคาเปิด)

In [ ]:
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# --- 1. วิเคราะห์อัตราส่วนความผันผวนเทียบราคาเปิด กลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=calculate_volatility_to_open,
    feature_col='Volatility_to_Open',
    feature_label_th='อัตราส่วนความผันผวนเทียบราคาเปิด (Volatility to Open)',
    title='การวิเคราะห์ความผันผวนเทียบราคาเปิด (Volatility to Open) ของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 2. วิเคราะห์อัตราส่วนความผันผวนเทียบราคาเปิด กลุ่มหุ้นส่งออก ---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=calculate_volatility_to_open,
    feature_col='Volatility_to_Open',
    feature_label_th='อัตราส่วนความผันผวนเทียบราคาเปิด (Volatility to Open)',
    title='การวิเคราะห์ความผันผวนเทียบราคาเปิด (Volatility to Open) ของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

### calculate_relative_return (Relative Return vs USD_THB)

In [ ]:
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# เตรียมข้อมูลอัตราแลกเปลี่ยน
df_fx_clean = clean_pct_series(df_USD_THB)

# ประยุกต์ใช้ฟังก์ชันคำนวณ Relative Return ก่อนนำไปพล็อต
def calc_rel_return_vs_usd_thb(df_stock):
    return calculate_relative_return(df_stock, df_fx_clean, factor_name='USD_THB')

# --- 1. วิเคราะห์ Relative Return กลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=calc_rel_return_vs_usd_thb,
    feature_col='Relative_Return_vs_USD_THB',
    feature_label_th='ผลตอบแทนสัมพัทธ์เทียบ USD/THB (Relative Return vs USD_THB)',
    title='การวิเคราะห์ Relative Return เทียบ USD/THB ของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 2. วิเคราะห์ Relative Return กลุ่มหุ้นส่งออก ---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=calc_rel_return_vs_usd_thb,
    feature_col='Relative_Return_vs_USD_THB',
    feature_label_th='ผลตอบแทนสัมพัทธ์เทียบ USD/THB (Relative Return vs USD_THB)',
    title='การวิเคราะห์ Relative Return เทียบ USD/THB ของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

### calculate_yield_slope (ความชันของเส้นอัตราผลตอบแทนพันธบัตรไทย: TH_20Y - TH_10Y)

In [ ]:
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# 1. คำนวณ Yield Curve Slope (TH_20Y - TH_10Y)
df_slope = calculate_yield_slope(df_TH_20Y, df_TH_10Y, long_name='Yield_20Y', short_name='Yield_10Y')

# 2. ฟังก์ชัน helper เพื่อรวมข้อมูล Slope เข้ากับราคาปิดของหุ้นแต่ละตัว
def calc_yield_slope_vs_stock(df_stock):
    # ทำความสะอาดข้อมูลวันที่ก่อน
    df_stk_cp = df_stock.copy()
    df_stk_cp['Date'] = pd.to_datetime(df_stk_cp['Date'])

    # รวมกับข้อมูล Slope
    merged = pd.merge(df_stk_cp, df_slope[['Date', 'Yield_Curve_Slope']], on='Date', how='inner')
    return merged

# --- 3. วิเคราะห์ความสัมพันธ์ของ Yield Curve Slope กลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=calc_yield_slope_vs_stock,
    feature_col='Yield_Curve_Slope',
    feature_label_th='ความชันพันธบัตรไทย (Yield Curve Slope: 20Y - 10Y)',
    title='การวิเคราะห์ Yield Curve Slope เทียบกับราคาปิดของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 4. วิเคราะห์ความสัมพันธ์ของ Yield Curve Slope กลุ่มหุ้นส่งออก ---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=calc_yield_slope_vs_stock,
    feature_col='Yield_Curve_Slope',
    feature_label_th='ความชันพันธบัตรไทย (Yield Curve Slope: 20Y - 10Y)',
    title='การวิเคราะห์ Yield Curve Slope เทียบกับราคาปิดของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

### calculate_relative_strength (อัตราส่วนความแข็งแกร่งเปรียบเทียบเชิงสัมพัทธ์: Export Stock / Import Stock)

**รายละเอียดและคำอธิบายสูตร:**
วิเคราะห์อัตราส่วนระหว่างราคาปิดของกลุ่มหุ้นส่งออกเปรียบเทียบกับกลุ่มหุ้นนำเข้าที่เป็นคู่ค้าหรืออยู่ในสเปกตรัมธุรกิจที่ตรงกันข้ามกัน เพื่อหาว่าเมื่อราคาของหุ้นตัวหนึ่งแข็งแกร่งขึ้น จะส่งผลต่อระดับราคาและความสัมพันธ์ของอีกตัวหนึ่งอย่างไร

* **Relative_Strength_Ratio** = ราคาปิดหุ้นส่งออก (Export Close) / ราคาปิดหุ้นนำเข้า (Import Close)
* **Return_Diff_Export_vs_Import** = เปอร์เซ็นต์การเปลี่ยนแปลงของหุ้นส่งออก - เปอร์เซ็นต์การเปลี่ยนแปลงของหุ้นนำเข้า (คำนวณเพื่อใช้ดูในมิติ % Change)

In [ ]:
# 1. กำหนดคู่หุ้นส่งออกและหุ้นนำเข้าเพื่อนำมาจับคู่เปรียบเทียบวิเคราะห์เชิงสัมพัทธ์
stock_pairs = [
    ('DELTA', 'PTTGC'),
    ('HANA', 'COM7'),
    ('KCE', 'IVL'),
    ('TU', 'CPF'),
    ('STA', 'BCP')
]

# 2. ฟังก์ชันประยุกต์ใช้เพื่อรวมคำนวณสำหรับสร้าง Scatter Plot
def calc_relative_strength_for_pair(df_export, df_import, exp_name, imp_name):
    df_pair = calculate_relative_strength(df_export, df_import, exp_stock_name=exp_name, imp_stock_name=imp_name)
    # คืนค่า DataFrame พร้อมกำหนดคอลัมน์หลักเป็นราคาปิดฝั่งส่งออกและคอลัมน์ฟีเจอร์เป็น อัตราส่วนความแข็งแกร่ง
    df_pair = df_pair.rename(columns={'Export_Close': 'Close'})
    return df_pair

# 3. สร้างโครงสร้าง dictionary สำหรับส่งผ่านพล็อตเตอร์
pairs_data_dict = {}
for exp_name, imp_name in stock_pairs:
    df_exp = globals()[f'df_{exp_name}']
    df_imp = globals()[f'df_{imp_name}']
    label = f'{exp_name} vs {imp_name}'
    # บันทึกเป็น lambda ให้สอดคล้องกับพารามิเตอร์ของฟังก์ชันยืดหยุ่น
    pairs_data_dict[label] = (df_exp, df_imp, exp_name, imp_name)

# ปรับฟังก์ชันแสดงผลเพื่อให้ทำงานกับข้อมูลโครงสร้างคู่หุ้น
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np

def plot_relative_strength_pairs_scatter(pairs_dict, color='#1f77b4'):
    thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
    sns.set_theme(style="whitegrid")

    num_pairs = len(pairs_dict)
    num_cols = 2
    num_rows = int(np.ceil(num_pairs / num_cols))

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 4.8 * num_rows))
    axes = axes.flatten() if num_pairs > 1 else np.array([axes])

    summary_list = []

    for idx, (label, (df_exp, df_imp, exp_name, imp_name)) in enumerate(pairs_dict.items()):
        df_feat = calculate_relative_strength(df_exp, df_imp, exp_stock_name=exp_name, imp_stock_name=imp_name)
        df_feat = df_feat.dropna(subset=['Relative_Strength_Ratio', 'Export_Close'])

        ratio_series = pd.to_numeric(df_feat['Relative_Strength_Ratio'], errors='coerce')
        close_series = pd.to_numeric(df_feat['Export_Close'], errors='coerce')

        correlation = ratio_series.corr(close_series)

        summary_list.append({
            'คู่หุ้น (Export vs Import)': label,
            'ค่าเฉลี่ย Relative Strength Ratio': ratio_series.mean(),
            'Correlation (vs Export Close)': correlation
        })

        sns.regplot(
            data=df_feat,
            x='Export_Close',
            y='Relative_Strength_Ratio',
            ax=axes[idx],
            scatter_kws={'alpha': 0.25, 'color': color, 's': 10},
            line_kws={'color': '#d62728', 'linewidth': 1.5}
        )

        axes[idx].set_title(f'{label} (Corr = {correlation:.3f})', fontproperties=thai_font, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(f'ราคาปิดของ {exp_name}', fontproperties=thai_font, fontsize=11)
        axes[idx].set_ylabel('อัตราความแข็งแกร่ง (Relative Strength Ratio)', fontproperties=thai_font, fontsize=11)

    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle('การวิเคราะห์ Relative Strength Ratio คู่หุ้นส่งออกเปรียบเทียบกับหุ้นนำเข้า', fontproperties=thai_font, fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    df_summary = pd.DataFrame(summary_list)
    print("\n=== [ตารางสรุปผล] ความสัมพันธ์ระหว่าง ราคาปิด และ Relative Strength Ratio ===")
    display(df_summary)

# ดำเนินการพล็อตกราฟ
plot_relative_strength_pairs_scatter(pairs_data_dict, color='#9467bd')

### convert_price_to_usd (การแปลงราคาปิดหุ้นจากสกุลเงินบาทเป็นดอลลาร์สหรัฐ: THB -> USD)

**รายละเอียดและคำอธิบายสูตร:**
วิเคราะห์ความสัมพันธ์ระหว่างราคาปิดของหุ้นในหน่วยดอลลาร์สหรัฐ (USD) กับราคาปิดสกุลเงินบาท (THB) เพื่อดูทิศทางและความแข็งแกร่งเมื่อรับรู้ผลกระทบจากความผันผวนของอัตราแลกเปลี่ยนโดยตรง

* **Stock_Close_USD** = ราคาปิดหุ้นสกุลบาท (Stock_Close_THB) / อัตราแลกเปลี่ยน USD_THB (USD_THB_Close)

In [ ]:
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# ฟังก์ชันประยุกต์ใช้เพื่อแปลงข้อมูลสำหรับส่งเข้าฟังก์ชันแสดงผล
def calc_close_usd(df_stock):
    return convert_price_to_usd(df_stock, df_USD_THB)

# --- 1. วิเคราะห์ราคาหุ้นปิดสกุลดอลลาร์สหรัฐ (Stock Close USD) ของกลุ่มหุ้นนำเข้า ---
plot_engineered_feature_scatter(
    stocks_dict=import_stocks_dict,
    feature_formula=calc_close_usd,
    feature_col='Stock_Close_USD',
    feature_label_th='ราคาปิดหน่วยดอลลาร์สหรัฐ (Stock Close USD)',
    title='การวิเคราะห์ราคาปิดในหน่วย USD (Stock Close USD) ของหุ้นกลุ่มนำเข้า',
    color='#2ca02c'
)

# --- 2. วิเคราะห์ราคาหุ้นปิดสกุลดอลลาร์สหรัฐ (Stock Close USD) ของกลุ่มหุ้นส่งออก ---
plot_engineered_feature_scatter(
    stocks_dict=export_stocks_dict,
    feature_formula=calc_close_usd,
    feature_col='Stock_Close_USD',
    feature_label_th='ราคาปิดหน่วยดอลลาร์สหรัฐ (Stock Close USD)',
    title='การวิเคราะห์ราคาปิดในหน่วย USD (Stock Close USD) ของหุ้นกลุ่มส่งออก',
    color='#1f77b4'
)

# ข้อมูลดัชนีและอัตราแลกเปลี่ยนเสริม (Phase 2 Expanded Features)
เชื่อมต่อข้อมูลดัชนีตลาดโลก (`SET50`, `SP500`, `SOX`, `VIX`, `DXY`, `BDI`) และอัตราแลกเปลี่ยนทางเลือก (`CNY_THB`, `EUR_THB`, `JPY_THB`, `USD_THB`) ที่ผ่านการคลีนจาก Step 1 เข้ามาสกัดฟีเจอร์เพิ่มเติม

### จัดเตรียมดิกชันนารีข้อมูลดัชนีและอัตราแลกเปลี่ยนสำหรับ Feature Engineering

### จัดเตรียมดิกชันนารีข้อมูลดัชนีและอัตราแลกเปลี่ยนสำหรับ Feature Engineering

In [ ]:
# แมปข้อมูลอัตราแลกเปลี่ยนและดัชนีที่คลีนแล้วจาก data/02_intermediate/
new_fx_dfs = {
    'df_new_fx_USD_THB': globals().get('df_USD_THB'),
    'df_new_fx_EUR_THB': globals().get('df_EUR_THB'),
    'df_new_fx_JPY_THB': globals().get('df_JPY_THB'),
    'df_new_fx_CNY_THB': globals().get('df_CNY_THB')
}

# แมปชื่อตัวแปรดัชนี
df_new_SET50 = globals().get('df_SET50')
df_new_SP500 = globals().get('df_SP500')
df_new_SOX = globals().get('df_SOX')
df_new_VIX = globals().get('df_VIX')
df_new_DXY = globals().get('df_DXY')
df_new_BDI = globals().get('df_BDI')

print("✓ แมปข้อมูลอัตราแลกเปลี่ยนและดัชนีเสริมเรียบร้อยแล้ว พร้อมสำหรับการสร้างฟีเจอร์")


# การรวมและจัดแนวข้อมูลตั้งต้น (Baseline Data Integration & Alignment)
คัดเลือกหุ้นนำเข้า 8 ตัว และหุ้นส่งออก 8 ตัว คำนวณผลตอบแทนเฉลี่ย `Import_Avg` และ `Export_Avg` พร้อมผสานฟีเจอร์ตั้งต้น 5 ปัจจัย (`Brent`, `US_10Y`, `USD_THB`, `Volatility_to_Open`, `Yield_Curve_Slope`) บันทึกเป็นชุดข้อมูลตั้งต้นใน `data/03_processed/baseline_pipeline_data.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. คัดเลือกรายชื่อหุ้น (คัดออกกลุ่มละ 2 ตัวตามบรีฟการวิจัย)
import_8_names = ['PTTGC', 'TOP', 'IRPC', 'IVL', 'GPSC', 'EGCO', 'RATCH', 'BCP']
export_8_names = ['DELTA', 'HANA', 'KCE', 'CPF', 'TU', 'STA', 'BDMS', 'BH']

def get_clean_return_and_close(df_name):
    df = globals()[df_name].copy()
    df['Date'] = pd.to_datetime(df['Date'])
    if 'Pct_Change' in df.columns:
        if df['Pct_Change'].dtype == 'object':
            df['Return'] = df['Pct_Change'].str.rstrip('%').astype('float') / 100.0
        else:
            df['Return'] = pd.to_numeric(df['Pct_Change'], errors='coerce')
    else:
        df = df.sort_values('Date')
        df['Return'] = df['Close'].pct_change()
    for c in ['Close', 'Open', 'High', 'Low']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df[['Date', 'Close', 'Open', 'High', 'Low', 'Return']]

import_dfs = {name: get_clean_return_and_close(f'df_{name}') for name in import_8_names}
export_dfs = {name: get_clean_return_and_close(f'df_{name}') for name in export_8_names}

# 2. คำนวณผลตอบแทนเฉลี่ยรายวันของกลุ่มนำเข้า (Import_Avg) และส่งออก (Export_Avg)
import_pct_merged = pd.DataFrame({'Date': pd.to_datetime(df_USD_THB['Date'])})
for name, df_stk in import_dfs.items():
    import_pct_merged = pd.merge(import_pct_merged, df_stk[['Date', 'Return']].rename(columns={'Return': name}), on='Date', how='left')
import_pct_merged['Import_Avg'] = import_pct_merged[import_8_names].mean(axis=1)

export_pct_merged = pd.DataFrame({'Date': pd.to_datetime(df_USD_THB['Date'])})
for name, df_stk in export_dfs.items():
    export_pct_merged = pd.merge(export_pct_merged, df_stk[['Date', 'Return']].rename(columns={'Return': name}), on='Date', how='left')
export_pct_merged['Export_Avg'] = export_pct_merged[export_8_names].mean(axis=1)

# 3. เตรียมฟีเจอร์มหภาคพื้นฐาน (X Factors)
brent_source = 'df_เบรนท์' if 'df_เบรนท์' in globals() else 'df_Brent'
df_brent_clean = get_clean_return_and_close(brent_source)[['Date', 'Return']].rename(columns={'Return': 'Brent'})
df_us10y_clean = get_clean_return_and_close('df_US_10Y')[['Date', 'Return']].rename(columns={'Return': 'US_10Y'})
df_usdthb_clean = get_clean_return_and_close('df_USD_THB')[['Date', 'Return']].rename(columns={'Return': 'USD_THB'})

# คำนวณ Volatility_to_Open เฉลี่ยของทั้ง 16 หุ้นในระบบ
vol_merged = pd.DataFrame({'Date': pd.to_datetime(df_USD_THB['Date'])})
vol_cols = []
for name, df_stk in {**import_dfs, **export_dfs}.items():
    df_stk['Vol_Open'] = (df_stk['High'] - df_stk['Low']) / df_stk['Open']
    vol_merged = pd.merge(vol_merged, df_stk[['Date', 'Vol_Open']].rename(columns={'Vol_Open': f'Vol_Open_{name}'}), on='Date', how='left')
    vol_cols.append(f'Vol_Open_{name}')
vol_merged['Volatility_to_Open'] = vol_merged[vol_cols].mean(axis=1)

# คำนวณ Yield_Curve_Slope (TH_20Y - TH_10Y)
df_slope_clean = calculate_yield_slope(df_TH_20Y, df_TH_10Y)[['Date', 'Yield_Curve_Slope']]

# 4. รวมข้อมูลเป็น final_pipeline_df
base_df = pd.DataFrame({'Date': sorted(pd.to_datetime(df_USD_THB['Date']).unique())})
final_pipeline_df = base_df.copy()
final_pipeline_df = pd.merge(final_pipeline_df, df_brent_clean, on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, df_us10y_clean, on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, df_usdthb_clean, on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, vol_merged[['Date', 'Volatility_to_Open']], on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, df_slope_clean, on='Date', how='inner')

# รวมตัวแปรเป้าหมาย Y
final_pipeline_df = pd.merge(final_pipeline_df, import_pct_merged[['Date', 'Import_Avg']], on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, export_pct_merged[['Date', 'Export_Avg']], on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, export_dfs['BH'][['Date', 'Return']].rename(columns={'Return': 'BH_Pct'}), on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, export_dfs['BDMS'][['Date', 'Return']].rename(columns={'Return': 'BDMS_Pct'}), on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, import_dfs['EGCO'][['Date', 'Return']].rename(columns={'Return': 'EGCO_Pct'}), on='Date', how='inner')
final_pipeline_df = pd.merge(final_pipeline_df, import_dfs['BCP'][['Date', 'Return']].rename(columns={'Return': 'BCP_Pct'}), on='Date', how='inner')

final_pipeline_df = final_pipeline_df.sort_values(by='Date').ffill().dropna().reset_index(drop=True)

# บันทึก baseline dataset ลงใน data/03_processed/
proc_dir = Path('../data/03_processed')
if not proc_dir.parent.exists():
    proc_dir = Path('data/03_processed')
proc_dir.mkdir(parents=True, exist_ok=True)
final_pipeline_df.to_csv(proc_dir / 'baseline_pipeline_data.csv', index=False)
print(f"✓ สร้างและบันทึก baseline_pipeline_data.csv สำเร็จ: {final_pipeline_df.shape[0]} วัน x {final_pipeline_df.shape[1]} คอลัมน์")
display(final_pipeline_df.head(5))


###แยกตัวแปรต้นตั้งต้น (Brent, US_10Y, USD_THB, Volatility_to_Open, Yield_Curve_Slope) และตัวแปรเป้าหมาย (Y) ออกจากกัน

In [ ]:
import pandas as pd

# 1. ตรวจสอบและดึงข้อมูลจาก final_pipeline_df
if 'final_pipeline_df' in globals():
    # แยกคอลัมน์ตัวแปรต้นตั้งต้น 5 ตัวแปร
    x_cols_base = ['Brent', 'US_10Y', 'USD_THB', 'Volatility_to_Open', 'Yield_Curve_Slope']
    X_base = final_pipeline_df[x_cols_base].copy()

    # แยกคอลัมน์ตัวแปรเป้าหมาย 6 ตัวแปร
    y_cols_all = ['Import_Avg', 'Export_Avg', 'BH_Pct', 'BDMS_Pct', 'EGCO_Pct', 'BCP_Pct']
    Y_all = final_pipeline_df[y_cols_all].copy()

    # เก็บข้อมูลคอลัมน์วันที่แยกไว้ต่างหากเพื่อรักษาลำดับเวลา
    dates = final_pipeline_df['Date'].copy()

    # 2. แสดงผลตรวจสอบขนาดของข้อมูล
    print("=== ตรวจสอบมิติข้อมูล (Shapes) ===")
    print(f"ขนาดของ X_base (Features): {X_base.shape}")
    print(f"ขนาดของ Y_all (Targets):  {Y_all.shape}")
    print(f"ขนาดของ dates (Temporal):  {dates.shape}\n")

    # 3. แสดงตัวอย่างข้อมูล 5 แถวแรก
    print("=== ตัวอย่างข้อมูล X_base (5 แถวแรก) ===")
    display(X_base.head(5))

    print("\n=== ตัวอย่างข้อมูล Y_all (5 แถวแรก) ===")
    display(Y_all.head(5))
else:
    print("ไม่พบตัวแปร 'final_pipeline_df' ในระบบ กรุณารันเซลล์จัดเตรียมข้อมูล (Pipeline) ก่อนหน้านี้")

###รวบทั้ง 3 กลุ่มฟีเจอร์ (Lagged 5 ตัว + Percentage Change/Momentum 3 ตัว + Rolling Volatility 5 วัน 2 ตัว)

In [ ]:
import pandas as pd
import numpy as np

# 1. สร้างตัวแปร X_fe โดยการคัดลอกจาก X_base
X_fe = X_base.copy()

# 2. สร้าง Lagged Features (5 ตัว)
X_fe['Brent_lag1'] = X_fe['Brent'].shift(1)
X_fe['Brent_lag2'] = X_fe['Brent'].shift(2)
X_fe['USD_THB_lag1'] = X_fe['USD_THB'].shift(1)
X_fe['USD_THB_lag2'] = X_fe['USD_THB'].shift(2)
X_fe['US_10Y_lag1'] = X_fe['US_10Y'].shift(1)

# 3. สร้าง Percentage Change / Momentum (3 ตัว) และจัดการแปลง inf เป็น NaN
X_fe['USD_THB_pct_1d'] = X_fe['USD_THB'].pct_change(1).replace([np.inf, -np.inf], np.nan)
X_fe['USD_THB_pct_5d'] = X_fe['USD_THB'].pct_change(5).replace([np.inf, -np.inf], np.nan)
X_fe['Brent_pct_5d'] = X_fe['Brent'].pct_change(5).replace([np.inf, -np.inf], np.nan)

# 4. สร้าง Rolling Volatility 5 วัน (2 ตัว)
X_fe['Volatility_roll5_std'] = X_fe['Volatility_to_Open'].rolling(window=5).std()
X_fe['USD_THB_roll5_std'] = X_fe['USD_THB'].rolling(window=5).std()

# 5. ตรวจสอบจำนวนค่าว่าง (NaN) ในแต่ละคอลัมน์ของ X_fe
print("=== ตรวจสอบค่าว่าง (NaN) ใน X_fe ===")
print(X_fe.isnull().sum())
print()

# 6. แสดงผลลัพธ์ 5 แถวแรกของ X_fe
print("=== ตัวอย่างข้อมูล X_fe (5 แถวแรก) ===")
display(X_fe.head(5))

###หลังจากล้างค่า NaN และจัดแนว Index ของ X_feat, Y_feat และ dates_feat

In [ ]:
import pandas as pd
import numpy as np

# 1. แปลงค่า np.inf และ -np.inf ใน X_fe ให้เป็น np.nan อีกครั้งเพื่อความปลอดภัย
X_fe_clean = X_fe.replace([np.inf, -np.inf], np.nan)

# 2. หา index ของแถวที่ไม่มีค่า NaN เลยใน X_fe_clean
valid_idx = X_fe_clean.dropna().index

# 3. กรองข้อมูลให้เหลือเฉพาะแถวที่สมบูรณ์และรีเซ็ต index ของทุกตัวแปร
X_feat = X_fe_clean.loc[valid_idx].reset_index(drop=True)
Y_feat = Y_all.loc[valid_idx].reset_index(drop=True)
dates_feat = dates.loc[valid_idx].reset_index(drop=True)

# 4. ตรวจสอบความถูกต้องและแสดงผลลัพธ์
print("=== [ผลการจัดการค่าว่างและการสร้างชุดฟีเจอร์สมบูรณ์ (X_feat)] ===\n")
print(f"จำนวนฟีเจอร์ทั้งหมดใน X_feat: {X_feat.shape[1]} คอลัมน์")
print(f"รายชื่อฟีเจอร์ทั้งหมด: {list(X_feat.columns)}\n")

print("=== ตรวจสอบมิติข้อมูล (Shapes) ===")
print(f"ขนาดของ X_feat (Features): {X_feat.shape}")
print(f"ขนาดของ Y_feat (Targets):  {Y_feat.shape}")
print(f"ขนาดของ dates_feat:         {dates_feat.shape}")
print("✓ จำนวนแถวของข้อมูลทุกตัวแปรตรงกันเรียบร้อยแล้ว" if len(X_feat) == len(Y_feat) == len(dates_feat) else "⚠ คำเตือน: จำนวนแถวของข้อมูลไม่เท่ากัน")
print()

# ตรวจสอบหาค่า NaN หรือ inf ที่อาจหลงเหลือ
remaining_nulls = X_feat.isnull().sum().sum()
remaining_infs = np.isinf(X_feat.select_dtypes(include=[np.number])).sum().sum()
print(f"จำนวนค่า NaN คงเหลือใน X_feat: {remaining_nulls} ค่า")
print(f"จำนวนค่า Infinity คงเหลือใน X_feat: {remaining_infs} ค่า\n")

# ตรวจสอบช่วงวันที่ของข้อมูล
print("=== ตรวจสอบช่วงเวลาของข้อมูล (Temporal Range) ===")
print(f"วันเริ่มต้น: {dates_feat.min().strftime('%Y-%m-%d')}")
print(f"วันสิ้นสุด: {dates_feat.max().strftime('%Y-%m-%d')}\n")

# แสดงข้อมูล 5 แถวแรกของ X_feat
print("=== ตัวอย่างข้อมูล X_feat (5 แถวแรก) ===")
display(X_feat.head(5))

### คำนวณ Correlation Matrix ระหว่างฟีเจอร์ทั้ง 15 ตัวใน `X_feat` (`X_feat.corr()`) แล้วพล็อต Heatmap

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ตั้งค่าฟอนต์ภาษาไทยสำหรับการแสดงผล
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
sns.set_theme(style="white")
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

# =====================================================================
# 1. พล็อต Heatmap ของฟีเจอร์ทั้ง 15 ตัวใน X_feat
# =====================================================================
corr_features = X_feat.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_features,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title("Correlation Heatmap of All 15 Features (X_feat)", fontproperties=thai_font, fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# =====================================================================
# 2. พล็อต Heatmap แสดงความสัมพันธ์ระหว่าง 15 ฟีเจอร์ กับ Targets (Y_feat)
# =====================================================================
# ผนวกข้อมูลและหา Correlation Matrix เฉพาะจุดเชื่อมโยงข้ามกลุ่ม
combined_df = pd.concat([X_feat, Y_feat], axis=1)
corr_features_vs_targets = combined_df.corr().loc[X_feat.columns, Y_feat.columns]

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_features_vs_targets,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-0.5,
    vmax=0.5,
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title("Correlation Heatmap: 15 Features vs Targets (Y_feat)", fontproperties=thai_font, fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# =====================================================================
# 3. ค้นหาและพิมพ์คู่ฟีเจอร์ที่มีความสัมพันธ์สูงมาก (|r| >= 0.85)
# =====================================================================
print("=== [รายงานคู่ฟีเจอร์ที่มีความสัมพันธ์กันสูงมาก (|r| >= 0.85)] ===\n")

# สร้าง Mask เพื่อดูเฉพาะครึ่งบนของ Matrix (เลี่ยงคู่ที่ซ้ำกันและตัวเอง)
upper_tri = corr_features.where(np.triu(np.ones(corr_features.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper_tri.columns:
    for row in upper_tri.index:
        val = upper_tri.loc[row, col]
        if not pd.isna(val) and abs(val) >= 0.85:
            high_corr_pairs.append((row, col, val))

if high_corr_pairs:
    print(f"พบคู่ฟีเจอร์ที่ซ้ำซ้อนสูงทั้งหมด {len(high_corr_pairs)} คู่ ดังนี้:")
    for idx, (f1, f2, r_val) in enumerate(high_corr_pairs, 1):
        print(f"  {idx}. {f1} <-> {f2}  (r = {r_val:+.3f})")
    print("\n💡 ข้อแนะนำ: คู่ฟีเจอร์ข้างต้นอาจส่งผลให้เกิดปัญหา Multicollinearity ควรพิจารณาเลือกใช้ตัวแปรใดตัวแปรหนึ่งในการทำนาย")
else:
    print("✓ ไม่พบคู่ฟีเจอร์ที่มีความสัมพันธ์กันเองเกิน 0.85 (ฟีเจอร์เป็นอิสระต่อกันดี)")

###Feature Selection & Statistical Validation (การตรวจสอบสัญญาณทางสถิติของตัวแปรต้นบน Train Set)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.feature_selection import mutual_info_classif
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# 1. ทำ Temporal Split ที่ 80% เช่นเดิมเพื่อป้องกัน Data Leakage
split_ratio = 0.8
split_idx = int(len(X_feat) * split_ratio)

X_train = X_feat.iloc[:split_idx].reset_index(drop=True)
Y_train = Y_feat.iloc[:split_idx].reset_index(drop=True)

# แปลง Target ของชุด Train เป็น Binary Classification
Y_train_class = Y_train.apply(lambda x: np.where(x > 0, 1, 0))

targets_to_analyze = ['Import_Avg', 'Export_Avg']
feature_names = X_train.columns

thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

# 2. คำนวณค่าทางสถิติทีละ Target บน Train Set เท่านั้น
for target in targets_to_analyze:
    y_tr = Y_train_class[target]

    # คำนวณ Mutual Information
    mi_scores = mutual_info_classif(X_train, y_tr, random_state=42)

    stats_results = []
    for idx, col in enumerate(feature_names):
        feat_series = X_train[col]

        # คำนวณ Spearman Correlation
        corr_coef, _ = spearmanr(feat_series, y_tr)

        # ทดสอบทางสถิติด้วย Mann-Whitney U Test (คลาส 0 vs คลาส 1)
        group0 = feat_series[y_tr == 0]
        group1 = feat_series[y_tr == 1]

        if len(group0) > 0 and len(group1) > 0:
            u_stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
        else:
            p_value = np.nan

        status = "Significant" if p_value < 0.05 else "Not Significant"

        stats_results.append({
            'Feature': col,
            'Spearman_Corr': corr_coef,
            'Mutual_Information': mi_scores[idx],
            'p-value (MWU)': p_value,
            'Status (alpha=0.05)': status
        })

    df_stats = pd.DataFrame(stats_results).sort_values(by='Mutual_Information', ascending=False).reset_index(drop=True)

    print(f"\n=== [ตารางสรุป Feature Selection & Statistical Validation] Target: {target} ===")
    display(df_stats)

    # 3. พล็อตกราฟเปรียบเทียบสำหรับแต่ละ Target
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # กราฟแท่ง Mutual Information
    sns.barplot(data=df_stats, x='Mutual_Information', y='Feature', ax=ax1, palette='viridis', hue='Feature', legend=False)
    ax1.set_title(f'Mutual Information Scores: {target}', fontproperties=thai_font, fontsize=12, fontweight='bold')
    ax1.set_xlabel('MI Score', fontproperties=thai_font, fontsize=10)
    ax1.set_ylabel('Features', fontproperties=thai_font, fontsize=10)
    for tick in ax1.get_yticklabels():
        tick.set_fontproperties(thai_font)

    # กราฟแท่ง Spearman Correlation
    df_sorted_corr = df_stats.sort_values(by='Spearman_Corr', key=abs, ascending=False)
    sns.barplot(data=df_sorted_corr, x='Spearman_Corr', y='Feature', ax=ax2, palette='coolwarm', hue='Feature', legend=False)
    ax2.set_title(f'Spearman Correlation: {target}', fontproperties=thai_font, fontsize=12, fontweight='bold')
    ax2.set_xlabel('Correlation Coefficient', fontproperties=thai_font, fontsize=10)
    ax2.set_ylabel('', fontproperties=thai_font, fontsize=10)
    ax2.axvline(0, color='black', linestyle='--', linewidth=0.8)
    for tick in ax2.get_yticklabels():
        tick.set_fontproperties(thai_font)

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. คัดเลือกเฉพาะ 8 ฟีเจอร์ที่กำหนดจาก X_feat
selected_features = ['Brent', 'US_10Y', 'USD_THB', 'Brent_lag1', 'Brent_lag2', 'USD_THB_lag1', 'USD_THB_lag2', 'USD_THB_pct_1d']
X_selected = X_feat[selected_features].copy()

# 2. แบ่งชุดข้อมูลแบบ Temporal Split ตามลำดับเวลา (80% Train, 20% Test)
split_ratio = 0.8
split_idx = int(len(X_selected) * split_ratio)

X_train_raw = X_selected.iloc[:split_idx].reset_index(drop=True)
X_test_raw = X_selected.iloc[split_idx:].reset_index(drop=True)

dates_train = dates_feat.iloc[:split_idx].reset_index(drop=True)
dates_test = dates_feat.iloc[split_idx:].reset_index(drop=True)

# 3. แปลงตัวแปรเป้าหมายใน Y_feat เป็น Binary Classification (1 เมื่อผลตอบแทน > 0, 0 เมื่อผลตอบแทน <= 0)
Y_train_cls = Y_feat.iloc[:split_idx].apply(lambda x: np.where(x > 0, 1, 0)).reset_index(drop=True)
Y_test_cls = Y_feat.iloc[split_idx:].apply(lambda x: np.where(x > 0, 1, 0)).reset_index(drop=True)

# 4. ทำ StandardScaler แบบป้องกัน Data Leakage
scaler = StandardScaler()

# fit และ transform เฉพาะบนชุด Train
X_train_scaled_arr = scaler.fit_transform(X_train_raw)
# transform บนชุด Test ด้วยพารามิเตอร์จากชุด Train เท่านั้น
X_test_scaled_arr = scaler.transform(X_test_raw)

# แปลงกลับเป็น DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled_arr, columns=selected_features)
X_test_scaled = pd.DataFrame(X_test_scaled_arr, columns=selected_features)

# 5. แสดงผลการตรวจสอบข้อมูล
print("=== 1. ตรวจสอบมิติข้อมูล (Shapes) ===")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")
print(f"Y_train_cls:    {Y_train_cls.shape}")
print(f"Y_test_cls:     {Y_test_cls.shape}")
print(f"dates_train:    {dates_train.shape}")
print(f"dates_test:     {dates_test.shape}\n")

print("=== 2. ตรวจสอบช่วงเวลาข้อมูล (Temporal Coverage) ===")
print(f"Train Set: เริ่มต้น {dates_train.min().strftime('%Y-%m-%d')} ถึง สิ้นสุด {dates_train.max().strftime('%Y-%m-%d')}")
print(f"Test Set:  เริ่มต้น {dates_test.min().strftime('%Y-%m-%d')} ถึง สิ้นสุด {dates_test.max().strftime('%Y-%m-%d')}")
# ตรวจสอบการทับซ้อนของวัน
overlap = set(dates_train).intersection(set(dates_test))
print(f"จำนวนวันที่ซ้อนทับกันระหว่าง Train และ Test: {len(overlap)} วัน\n")

print("=== 3. ตรวจสอบความสมบูรณ์ของข้อมูล (NaN & Infinity Check) ===")
train_nans = X_train_scaled.isnull().sum().sum()
test_nans = X_test_scaled.isnull().sum().sum()
train_infs = np.isinf(X_train_scaled.to_numpy()).sum()
test_infs = np.isinf(X_test_scaled.to_numpy()).sum()
print(f"พบค่า NaN ใน X_train_scaled: {train_nans} | ใน X_test_scaled: {test_nans}")
print(f"พบค่า Inf ใน X_train_scaled: {train_infs} | ใน X_test_scaled: {test_infs}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from IPython.display import display

# 1. ตั้งค่าฟอนต์ภาษาไทยสำหรับกราฟแสดงผล
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

# 2. นิยามโมเดลที่ต้องการฝึกสอนและประเมิน
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

all_results = []
best_models = {}

# 3. วนลูปเทรนและประเมินผลทีละตัวแปรเป้าหมาย (6 Targets)
for col in Y_train_cls.columns:
    y_tr = Y_train_cls[col]
    y_te = Y_test_cls[col]

    # --- A. Baseline Model (Majority Class) ---
    majority_class = y_tr.mode()[0]
    baseline_preds = np.full(shape=len(y_te), fill_value=majority_class)

    # คำนวณอัตราส่วนการทำนาย (ควรเป็น 100% สำหรับคลาสส่วนใหญ่)
    base_class_1_ratio = np.mean(baseline_preds == 1)
    base_pred_dist = f"1: {base_class_1_ratio:.1%}, 0: {1 - base_class_1_ratio:.1%}"

    all_results.append({
        'Target': col,
        'Model': 'Baseline (Majority Class)',
        'Accuracy': accuracy_score(y_te, baseline_preds),
        'Precision': precision_score(y_te, baseline_preds, zero_division=0),
        'Recall': recall_score(y_te, baseline_preds, zero_division=0),
        'F1-Score': f1_score(y_te, baseline_preds, zero_division=0),
        'Prediction Distribution': base_pred_dist
    })

    # --- B. Machine Learning Models ---
    best_acc = -1
    best_model_name = None
    best_clf_obj = None

    for model_name, clf in models.items():
        # ฝึกสอนโมเดลบนชุดข้อมูลที่ Scale แล้ว
        clf.fit(X_train_scaled, y_tr)
        preds = clf.predict(X_test_scaled)

        acc = accuracy_score(y_te, preds)
        prec = precision_score(y_te, preds, zero_division=0)
        rec = recall_score(y_te, preds, zero_division=0)
        f1 = f1_score(y_te, preds, zero_division=0)

        # คำนวณสัดส่วนการทายคลาส 1 และ คลาส 0
        class_1_ratio = np.mean(preds == 1)
        pred_dist = f"1: {class_1_ratio:.1%}, 0: {1 - class_1_ratio:.1%}"

        all_results.append({
            'Target': col,
            'Model': model_name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'Prediction Distribution': pred_dist
        })

        # เก็บโมเดลที่ดีที่สุดอ้างอิงจาก Accuracy บน Test Set เพื่อนำมาวิเคราะห์ Feature Importance
        if acc > best_acc:
            best_acc = acc
            best_model_name = model_name
            best_clf_obj = clf

    best_models[col] = (best_model_name, best_clf_obj)

# 4. แสดงตารางเปรียบเทียบผลลัพธ์
df_summary = pd.DataFrame(all_results)
print("=== [ตารางสรุปผลประสิทธิภาพโมเดลจำแนกตามเป้าหมายบน Test Set] ===")
for col in Y_train_cls.columns:
    print(f"\n ตัวแปรเป้าหมาย: {col}")
    df_target_sub = df_summary[df_summary['Target'] == col].sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
    display(df_target_sub)

# 5. แสดงผล Feature Importance หรือ Coefficients สำหรับโมเดลที่ดีที่สุดในแต่ละ Target
print("\n=== [การวิเคราะห์อิทธิพลของตัวแปรต้น (Feature Weight / Importance Analysis)] ===")
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, col in enumerate(Y_train_cls.columns):
    best_name, clf = best_models[col]
    ax = axes[idx]

    if best_name == 'Logistic Regression':
        # ใช้ค่า Coefficient
        importance_vals = clf.coef_[0]
        importance_series = pd.Series(importance_vals, index=selected_features).sort_values()
        sns.barplot(x=importance_series.values, y=importance_series.index, ax=ax, palette='RdBu_r', hue=importance_series.index, legend=False)
        ax.set_title(f"{col} Best Model: {best_name}\n(Coefficient Value)", fontproperties=thai_font, fontsize=11, fontweight='bold')
        ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    else:
        # ใช้ค่า Feature Importances สำหรับ Tree-based models
        importance_vals = clf.feature_importances_
        importance_series = pd.Series(importance_vals, index=selected_features).sort_values()
        sns.barplot(x=importance_series.values, y=importance_series.index, ax=ax, palette='viridis', hue=importance_series.index, legend=False)
        ax.set_title(f"{col} Best Model: {best_name}\n(Gini Feature Importance)", fontproperties=thai_font, fontsize=11, fontweight='bold')

    ax.set_xlabel('ระดับความสำคัญ / น้ำหนักของตัวแปร', fontproperties=thai_font, fontsize=9)
    for tick in ax.get_yticklabels():
        tick.set_fontproperties(thai_font)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

# 1. โหลดฟอนต์ภาษาไทยสำหรับการพล็อตกราฟ
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# 2. นิยามโครงสร้างโมเดล Logistic Regression
base_lr = LogisticRegression(max_iter=1000, random_state=42)
selected_features = ['Brent', 'US_10Y', 'USD_THB', 'Brent_lag1', 'Brent_lag2', 'USD_THB_lag1', 'USD_THB_lag2', 'USD_THB_pct_1d']

results_lr = []
coef_dict = {}
preds_dict = {}

# 3. เทรนและประเมินผลรายตัวแปรเป้าหมาย
for col in Y_train_cls.columns:
    y_tr = Y_train_cls[col]
    y_te = Y_test_cls[col]

    # คำนวณ Baseline (Majority Class)
    majority_val = y_tr.mode()[0]
    base_preds = np.full(shape=len(y_te), fill_value=majority_val)
    base_acc = accuracy_score(y_te, base_preds)
    base_f1 = f1_score(y_te, base_preds, zero_division=0)

    # เทรน Logistic Regression Clone
    model_lr = clone(base_lr)
    model_lr.fit(X_train_scaled, y_tr)
    preds_lr = model_lr.predict(X_test_scaled)

    lr_acc = accuracy_score(y_te, preds_lr)
    lr_f1 = f1_score(y_te, preds_lr, zero_division=0)

    # บันทึกสัมประสิทธิ์และผลลัพธ์การพยากรณ์
    coef_dict[col] = model_lr.coef_[0]
    preds_dict[col] = preds_lr

    # สรุปอัตราส่วนคลาสที่ทำนาย
    class_1_ratio = np.mean(preds_lr == 1)

    results_lr.append({
        'Target': col,
        'Baseline_Accuracy': base_acc,
        'LR_Accuracy': lr_acc,
        'Baseline_F1': base_f1,
        'LR_F1': lr_f1,
        'Pred_Class_1_Ratio': class_1_ratio
    })

df_lr_summary = pd.DataFrame(results_lr)
print("=== [ตารางสรุปเปรียบเทียบผลลัพธ์ Logistic Regression vs Baseline] ===")
display(df_lr_summary)

# =========================================================
# กราฟชุดที่ 1: เปรียบเทียบประสิทธิภาพ (Accuracy & F1-Score)
# =========================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x_indices = np.arange(len(df_lr_summary['Target']))
bar_width = 0.35

# พล็อต Accuracy
ax1.bar(x_indices - bar_width/2, df_lr_summary['Baseline_Accuracy'], width=bar_width, label='Baseline', color='#7f7f7f', alpha=0.7)
ax1.bar(x_indices + bar_width/2, df_lr_summary['LR_Accuracy'], width=bar_width, label='Logistic Regression', color='#1f77b4')
ax1.set_title('เปรียบเทียบความแม่นยำ (Accuracy) ของแต่ละเป้าหมาย', fontproperties=thai_font, fontsize=14, fontweight='bold')
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df_lr_summary['Target'], rotation=30)
ax1.set_ylabel('Accuracy Score', fontproperties=thai_font, fontsize=11)
ax1.legend(prop=thai_font)

# พล็อต F1-Score
ax2.bar(x_indices - bar_width/2, df_lr_summary['Baseline_F1'], width=bar_width, label='Baseline', color='#7f7f7f', alpha=0.7)
ax2.bar(x_indices + bar_width/2, df_lr_summary['LR_F1'], width=bar_width, label='Logistic Regression', color='#2ca02c')
ax2.set_title('เปรียบเทียบค่า F1-Score ของแต่ละเป้าหมาย', fontproperties=thai_font, fontsize=14, fontweight='bold')
ax2.set_xticks(x_indices)
ax2.set_xticklabels(df_lr_summary['Target'], rotation=30)
ax2.set_ylabel('F1-Score', fontproperties=thai_font, fontsize=11)
ax2.legend(prop=thai_font)

plt.tight_layout()
plt.show()

# =========================================================
# กราฟชุดที่ 2: น้ำหนักตัวแปร Coefficients Bar Chart แยกตาม 6 เป้าหมาย
# =========================================================
df_coefs = pd.DataFrame(coef_dict, index=selected_features)

fig2, axes2 = plt.subplots(3, 2, figsize=(16, 14), sharey=True)
axes2 = axes2.flatten()

for idx, col in enumerate(Y_train_cls.columns):
    importance = df_coefs[col].sort_values()
    sns.barplot(x=importance.values, y=importance.index, ax=axes2[idx], palette='RdBu_r', hue=importance.index, legend=False)
    axes2[idx].set_title(f'สัมประสิทธิ์ (Coefficients) สำหรับเป้าหมาย: {col}', fontproperties=thai_font, fontsize=12, fontweight='bold')
    axes2[idx].axvline(0, color='black', linestyle='--', linewidth=0.8)
    axes2[idx].set_xlabel('Coefficient Value', fontproperties=thai_font, fontsize=10)
    for tick in axes2[idx].get_yticklabels():
        tick.set_fontproperties(thai_font)

fig2.suptitle('น้ำหนักและทิศทางของตัวแปรต้นต่อเป้าหมายพยากรณ์แต่ละตัว', fontproperties=thai_font, fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# =========================================================
# กราฟชุดที่ 3: เปรียบเทียบทิศทางพยากรณ์ในช่วง dates_test (60 วันล่าสุด)
# =========================================================
plot_days = 60
test_dates_dt = pd.to_datetime(dates_test)

fig3, axes3 = plt.subplots(6, 1, figsize=(15, 18), sharex=True)

for idx, col in enumerate(Y_train_cls.columns):
    y_act = Y_test_cls[col].values[-plot_days:]
    y_prd = preds_dict[col][-plot_days:]
    dates_sub = test_dates_dt.values[-plot_days:]

    axes3[idx].step(dates_sub, y_act, where='mid', label='Actual Direction', color='#1f77b4', alpha=0.8, linewidth=2)
    axes3[idx].step(dates_sub, y_prd, where='mid', label='Predicted Direction', color='#d62728', linestyle='--', linewidth=1.8)
    axes3[idx].set_title(f'อนุกรมเวลาเปรียบเทียบทิศทางจริง vs คำทำนาย: {col} ({plot_days} วันล่าสุด)', fontproperties=thai_font, fontsize=11, fontweight='bold')
    axes3[idx].set_yticks([0, 1])
    axes3[idx].set_yticklabels(['ลง/คงที่ (0)', 'ขึ้น (1)'], fontproperties=thai_font, fontsize=9)
    axes3[idx].legend(prop=thai_font, loc='upper right')
    axes3[idx].grid(True, linestyle=':', alpha=0.5)

plt.xticks(rotation=45)
fig3.suptitle('เปรียบเทียบทิศทางการขึ้นลงจริงเทียบกับคำทำนายของ Logistic Regression ในชุดข้อมูลทดสอบ', fontproperties=thai_font, fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

###ทำความสะอาดข้อมูลอัตราแลกเปลี่ยน

In [ ]:
import pandas as pd

# 1. กำหนดกลุ่มข้อมูลอัตราแลกเปลี่ยนชุดใหม่
new_fx_dfs = {
    'df_new_fx_USD_THB': df_new_fx_USD_THB,
    'df_new_fx_EUR_THB': df_new_fx_EUR_THB,
    'df_new_fx_JPY_THB': df_new_fx_JPY_THB,
    'df_new_fx_CNY_THB': df_new_fx_CNY_THB
}

print("=== [ขั้นตอนที่ 1] การตรวจสอบข้อมูลซ้ำซ้อน (Duplicate Checking) ===\n")
for name, df in new_fx_dfs.items():
    print(f"📌 ตรวจสอบข้อมูลซ้ำสำหรับ: {name}")
    check_duplicates(df)
    print("-" * 60)


In [ ]:
print("=== [ขั้นตอนที่ 2] การจัดการคอลัมน์ Volume และลบแถวที่มีค่าว่าง (Null Handling) ===\n")

for name, df in new_fx_dfs.items():
    print(f"📌 ตรวจสอบค่าว่างตั้งต้นของ: {name}")
    check_and_show_nulls(df)

    # ดรอปคอลัมน์ 'ปริมาณ' (Volume) ออกก่อน เนื่องจากอัตราแลกเปลี่ยนไม่มีปริมาณการซื้อขาย (เป็น NaN ทั้งหมด)
    df_cleaned = drop_columns(df, ['ปริมาณ'])

    # ลบแถวที่ยังมีค่าว่างอื่นๆ หลงเหลืออยู่
    df_final = drop_null_rows(df_cleaned)

    # บันทึกค่าที่ทำความสะอาดเสร็จแล้วกลับไปยัง Global variable
    globals()[name] = df_final
    print(f"✓ {name} ทำความสะอาดเสร็จสิ้นคงเหลือขนาด: {df_final.shape[0]} แถว x {df_final.shape[1]} คอลัมน์")
    print("-" * 60)


In [ ]:
# 1. กำหนดดิกชันนารีของอัตราแลกเปลี่ยนชุดใหม่ที่ต้องการลบคอลัมน์ ปริมาณ (Volume) และเคลียร์ค่าว่าง
new_fx_dfs = {
    'df_new_fx_USD_THB': df_new_fx_USD_THB,
    'df_new_fx_EUR_THB': df_new_fx_EUR_THB,
    'df_new_fx_JPY_THB': df_new_fx_JPY_THB,
    'df_new_fx_CNY_THB': df_new_fx_CNY_THB
}

print("=== เริ่มต้นทำความสะอาดข้อมูลอัตราแลกเปลี่ยนชุดใหม่ ===\n")

for name, df in new_fx_dfs.items():
    print(f"📌 ดำเนินการลบคอลัมน์ 'ปริมาณ' และลบแถวที่เป็นค่าว่างสำหรับ: {name}")

    # ลบคอลัมน์ 'ปริมาณ' (Volume) ออก
    df_no_volume = drop_columns(df, ['ปริมาณ'])

    # ลบแถวที่มีค่าว่างอื่น ๆ ในข้อมูล
    df_cleaned = drop_null_rows(df_no_volume)

    # อัปเดตตัวแปร Global ด้วย DataFrame ที่สะอาดเรียบร้อยแล้ว
    globals()[name] = df_cleaned
    print(f"✓ {name} ทำความสะอาดสำเร็จ คงเหลือขนาด: {df_cleaned.shape[0]} แถว x {df_cleaned.shape[1]} คอลัมน์")
    print("-" * 60)

###สร้าง df อัตราแลกเปลี่ยนใหม่

In [ ]:
import pandas as pd
from IPython.display import display

# แสดงตัวอย่างข้อมูล 5 แถวแรกของอัตราแลกเปลี่ยนใหม่ที่สะอาดแล้ว
print("=== ตัวอย่างข้อมูล df_new_fx_USD_THB ===")
display(df_new_fx_USD_THB.head(5))

print("\n=== ตัวอย่างข้อมูล df_new_fx_EUR_THB ===")
display(df_new_fx_EUR_THB.head(5))

print("\n=== ตัวอย่างข้อมูล df_new_fx_JPY_THB ===")
display(df_new_fx_JPY_THB.head(5))

print("\n=== ตัวอย่างข้อมูล df_new_fx_CNY_THB ===")
display(df_new_fx_CNY_THB.head(5))

###ทำความสะอาดข้อมูลดัชนี

In [ ]:
import pandas as pd

# 1. รวบรวมและดึงข้อมูลดัชนีชุดใหม่ทั้งหมดอย่างปลอดภัย
new_index_dfs = {
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_SET_50': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_SET_50'),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index'),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_S&P_500': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_S&P_500', globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_SP_500')),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_CBOE_Volatility_Index': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_CBOE_Volatility_Index'),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index_(1)': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index_(1)'),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Baltic_Dry_Index': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Baltic_Dry_Index'),
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Philadelphia_Semiconductor_Index': globals().get('df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Philadelphia_Semiconductor_Index'),
}

print("=== [ขั้นตอนที่ 1] ตรวจสอบข้อมูลซ้ำซ้อนสำหรับข้อมูลดัชนี ===\n")
for name, df in sorted(new_index_dfs.items()):
    if df is not None:
        print(f"📌 ตรวจสอบข้อมูลซ้ำสำหรับ: {name}")
        check_duplicates(df)
        print("-" * 60)
    else:
        print(f"⚠️ ไม่พบตัวแปรสำหรับ: {name}")
        print("-" * 60)

In [ ]:
print("=== [ขั้นตอนที่ 2] ตรวจสอบค่าว่างและทำความสะอาดสำหรับข้อมูลดัชนี ===\n")

for name, df in sorted(new_index_dfs.items()):
    print(f"📌 ดำเนินการตรวจสอบและทำความสะอาด: {name}")
    # ตรวจสอบว่ามีค่าว่างหรือไม่
    check_and_show_nulls(df)

    # ดรอปคอลัมน์ ปริมาณ (Volume) ออกหากมีค่าเป็น Null ทั้งหมดหรือเกือบทั้งหมด เพื่อป้องกันข้อผิดพลาด
    if 'ปริมาณ' in df.columns and df['ปริมาณ'].isnull().all():
        df_cleaned = drop_columns(df, ['ปริมาณ'])
    elif 'ปริมาณ' in df.columns and df['ปริมาณ'].isnull().sum() > 0:
        # สำหรับดัชนีที่คอลัมน์ปริมาณเป็น NaN ในบางวัน แต่เราจำเป็นต้องใช้ข้อมูลราคาปิด แนะนำให้ดรอปคอลัมน์นี้
        df_cleaned = drop_columns(df, ['ปริมาณ'])
    else:
        df_cleaned = df.copy()

    # ลบแถวที่มีค่าว่างอื่น ๆ เพื่อให้ข้อมูลสมบูรณ์
    df_final = drop_null_rows(df_cleaned)

    # บันทึกค่าที่ทำความสะอาดเสร็จกลับไปยัง Global variable
    globals()[name] = df_final
    print(f"✓ {name} ทำความสะอาดสำเร็จ คงเหลือขนาด: {df_final.shape[0]} แถว x {df_final.shape[1]} คอลัมน์")
    print("-" * 60)

In [ ]:
import pandas as pd

# 1. กำหนดรายชื่อดัชนีชุดใหม่ทั้งหมด
new_index_varnames = [
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_SET_50',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_S&P_500',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_CBOE_Volatility_Index',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index_(1)',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Baltic_Dry_Index',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Philadelphia_Semiconductor_Index'
]

print("=== เริ่มต้นการลบคอลัมน์ 'ปริมาณ' ออกจากข้อมูลดัชนีชุดใหม่ ===\n")

for name in new_index_varnames:
    df = globals().get(name)
    if df is not None:
        # ใช้ฟังก์ชัน drop_columns ที่กำหนดไว้ก่อนหน้าในการลบคอลัมน์อย่างปลอดภัย
        globals()[name] = drop_columns(df, ['ปริมาณ'])
        print(f"✓ {name} ปรับปรุงเรียบร้อย -> คอลัมน์คงเหลือ: {list(globals()[name].columns)}")
        print("-" * 60)


###สร้าง df ข้อมูลดัชนีใหม่

In [ ]:
import pandas as pd
from IPython.display import display

# 1. กำหนดแมปปิ้งชื่อตัวแปรดัชนีภาษาไทยเป็นภาษาอังกฤษมาตรฐานสำหรับชุดข้อมูลใหม่
index_variable_mapping = {
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_SET_50': 'df_new_SET50',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index': 'df_new_DXY',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_S&P_500': 'df_new_SP500',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_CBOE_Volatility_Index': 'df_new_VIX',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_US_Dollar_Index_(1)': 'df_new_DXY_alt',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Baltic_Dry_Index': 'df_new_BDI',
    'df_new_index_ข้อมูลย้อนหลังของ_ดัชนี_Philadelphia_Semiconductor_Index': 'df_new_SOX'
}

print("=== เริ่มสร้างตัวแปรมาตรฐานภาษาอังกฤษสำหรับข้อมูลดัชนีชุดใหม่ ===\n")

for thai_var, eng_var in index_variable_mapping.items():
    df = globals().get(thai_var)
    if df is not None:
        # จัดการแปลงชื่อคอลัมน์ให้อยู่ในมาตรฐานเดียวกัน (Date, Close, Open, High, Low, Pct_Change)
        column_mapping = {
            'วันเดือนปี': 'Date',
            'ล่าสุด': 'Close',
            'ราคาเปิด': 'Open',
            'สูงสุด': 'High',
            'ต่ำสุด': 'Low',
            '% เปลี่ยน': 'Pct_Change'
        }
        df_renamed = df.rename(columns=column_mapping)

        # จัดเรียงลำดับคอลัมน์ให้เป็นระเบียบ
        cols_order = ['Date', 'Close', 'Open', 'High', 'Low', 'Pct_Change']
        existing_cols = [c for c in cols_order if c in df_renamed.columns]
        df_renamed = df_renamed[existing_cols]

        # ประกาศตัวแปร Global ใหม่
        globals()[eng_var] = df_renamed
        print(f"✓ สร้างตัวแปร {eng_var} สำเร็จ! (ขนาด: {df_renamed.shape[0]} แถว x {df_renamed.shape[1]} คอลัมน์)")
    else:
        print(f"⚠ ไม่พบตัวแปร {thai_var} ในระบบ")


In [ ]:
# 2. แสดงตัวอย่างข้อมูลดัชนีชุดใหม่ที่ถูกจัดมาตรฐานเรียบร้อยแล้ว
print("=== ตัวอย่างข้อมูลดัชนีใหม่บางส่วน ===")
for eng_var in ['df_new_SET50', 'df_new_SP500', 'df_new_VIX', 'df_new_BDI', 'df_new_SOX']:
    df = globals().get(eng_var)
    if df is not None:
        print(f"\n• ตัวอย่างข้อมูลของ {eng_var}: ")
        display(df.head(3))


#หาความสัมพันธ์

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

# 1. ตั้งค่าฟอนต์ภาษาไทยสำหรับการแสดงผลบนกราฟ
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white")

# ฟังก์ชันดึงและทำความสะอาด Pct_Change หรือคำนวณ % Change รายวัน
def get_daily_return(df, name_col):
    df_clean = df.copy()

    # ตรวจสอบและแปลงคอลัมน์วันที่
    date_col = 'Date' if 'Date' in df_clean.columns else ('วันเดือนปี' if 'วันเดือนปี' in df_clean.columns else df_clean.columns[0])
    df_clean['Date_Parsed'] = pd.to_datetime(df_clean[date_col])
    df_clean = df_clean.sort_values('Date_Parsed')

    pct_col = 'Pct_Change' if 'Pct_Change' in df_clean.columns else ('% เปลี่ยน' if '% เปลี่ยน' in df_clean.columns else None)

    if pct_col and pct_col in df_clean.columns:
        if df_clean[pct_col].dtype == 'object':
            df_clean['Return'] = df_clean[pct_col].str.rstrip('%').astype('float') / 100.0
        else:
            df_clean['Return'] = pd.to_numeric(df_clean[pct_col], errors='coerce')
    else:
        close_col = 'Close' if 'Close' in df_clean.columns else ('ล่าสุด' if 'ล่าสุด' in df_clean.columns else df_clean.columns[1])
        df_clean['Return'] = pd.to_numeric(df_clean[close_col], errors='coerce').pct_change()

    return df_clean[['Date_Parsed', 'Return']].rename(columns={'Date_Parsed': 'Date', 'Return': name_col}).dropna()

# 2. ตรวจสอบและโหลด final_pipeline_df จากไฟล์หากไม่มีใน Memory
if 'final_pipeline_df' not in globals():
    data_candidates = [
        Path('../data/03_processed/final_pipeline_data.csv'),
        Path('data/03_processed/final_pipeline_data.csv'),
        Path('./final_pipeline_data.csv'),
        Path('/content/final_pipeline_data.csv'),
        Path('/content/drive/MyDrive/Colab Notebooks/DSDN/final_pipeline_data.csv')
    ]
    loaded_pipe = False
    for p in data_candidates:
        if p.exists():
            print(f"✓ โหลดข้อมูลสำเร็จจาก: {p}")
            final_pipeline_df = pd.read_csv(p)
            loaded_pipe = True
            break
    if not loaded_pipe:
        raise FileNotFoundError("ไม่พบไฟล์ข้อมูล final_pipeline_data.csv กรุณาย้อนกลับไปรันขั้นตอนจัดเตรียม Pipeline")

# 3. รวบรวมข้อมูลปัจจัยรายวันชุดใหม่
daily_factors = {
    'DXY': df_new_DXY,
    'SET50': df_new_SET50,
    'SP500': df_new_SP500,
    'VIX': df_new_VIX,
    'BDI': df_new_BDI,
    'SOX': df_new_SOX,
    'EUR_THB': df_new_fx_EUR_THB,
    'JPY_THB': df_new_fx_JPY_THB,
    'CNY_THB': df_new_fx_CNY_THB,
    'USD_THB': df_new_fx_USD_THB
}

# ทำการแปลงข้อมูลปัจจัยนำเข้าเป็น Daily Return
df_factors_merged = None
for name, df_fac in daily_factors.items():
    if df_fac is not None:
        df_ret = get_daily_return(df_fac, name)
        if df_factors_merged is None:
            df_factors_merged = df_ret
        else:
            df_factors_merged = pd.merge(df_factors_merged, df_ret, on='Date', how='inner')

# 4. เตรียมตัวแปรเป้าหมาย (Y) จาก final_pipeline_df
# ตรวจสอบชื่อคอลัมน์จริงใน DataFrame ที่บันทึกไว้
target_cols_y = [col for col in ['Import_Avg', 'Export_Avg', 'BH_Pct', 'BDMS_Pct', 'EGCO_Pct', 'BCP_Pct'] if col in final_pipeline_df.columns]

df_targets = final_pipeline_df[['Date'] + target_cols_y].copy()
df_targets['Date'] = pd.to_datetime(df_targets['Date'])

# 5. รวมข้อมูลทั้งหมดเข้าด้วยกัน
df_daily_analysis = pd.merge(df_factors_merged, df_targets, on='Date', how='inner')

# 6. คำนวณ Correlation Matrix
daily_corr = df_daily_analysis.drop(columns=['Date']).corr()
sub_daily_corr = daily_corr.loc[list(daily_factors.keys()), target_cols_y]

# 7. พล็อต Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    sub_daily_corr,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    fmt=".3f",
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title("ความสัมพันธ์รายวัน (Daily % Change Correlation)\nระหว่างปัจจัยนำเข้าใหม่ กับ ตัวแปรเป้าหมาย Y", fontproperties=thai_font, fontsize=14, fontweight='bold', pad=15)
plt.xticks(fontproperties=thai_font, fontsize=11, rotation=45, ha='right')
plt.yticks(fontproperties=thai_font, fontsize=11, rotation=0)
plt.tight_layout()
plt.show()

print("=== [ตารางสรุปผล] ค่าสหสัมพันธ์รายวันระหว่าง ปัจจัยนำเข้าใหม่ กับ ตัวแปรเป้าหมาย Y ===")
display(sub_daily_corr)

###เช็กขนาดข้อมูลหลังรวม (shape และช่วงวันที่)

In [ ]:
import pandas as pd
import numpy as np

# 1. มั่นใจว่าข้อมูลวันที่ของทั้งสอง DataFrame อยู่ในรูปแบบ Datetime
df_daily_analysis['Date'] = pd.to_datetime(df_daily_analysis['Date'])
final_pipeline_df['Date'] = pd.to_datetime(final_pipeline_df['Date'])

# 2. ตรวจสอบรายละเอียดเบื้องต้นของ df_daily_analysis
shape_daily = df_daily_analysis.shape
min_date_daily = df_daily_analysis['Date'].min()
max_date_daily = df_daily_analysis['Date'].max()
null_count_daily = df_daily_analysis.isnull().sum().sum()
duplicated_dates_daily = df_daily_analysis['Date'].duplicated().sum()

# 3. ตรวจสอบรายละเอียดเบื้องต้นของ final_pipeline_df
shape_pipe = final_pipeline_df.shape
min_date_pipe = final_pipeline_df['Date'].min()
max_date_pipe = final_pipeline_df['Date'].max()

# 4. วิเคราะห์เปรียบเทียบวันทำการที่หายไปจากการ Aligned (Inner Merge)
dates_pipe_set = set(final_pipeline_df['Date'])
dates_daily_set = set(df_daily_analysis['Date'])

lost_dates = dates_pipe_set - dates_daily_set
num_lost_days = len(lost_dates)
pct_lost_days = (num_lost_days / len(dates_pipe_set)) * 100

# 5. แสดงรายงานการวิเคราะห์เชิงลึก
print("=== [รายงานผลการวิเคราะห์ข้อมูลหลังทำ Temporal Inner Alignment] ===\n")
print(f"📊 ข้อมูล DataFrame: df_daily_analysis")
print(f"  - ขนาดของข้อมูล (Shape): {shape_daily[0]} แถว x {shape_daily[1]} คอลัมน์")
print(f"  - ช่วงเวลา: ตั้งแต่ {min_date_daily.strftime('%Y-%m-%d')} ถึง {max_date_daily.strftime('%Y-%m-%d')}")
print(f"  - จำนวนแถวที่มีวันที่ซ้ำกัน: {duplicated_dates_daily} แถว")
print(f"  - จำนวนค่าว่าง (NaN) ทั้งหมด: {null_count_daily} ตำแหน่ง\n")

print(f"📊 ข้อมูล DataFrame ดั้งเดิม: final_pipeline_df")
print(f"  - ขนาดของข้อมูล (Shape): {shape_pipe[0]} แถว x {shape_pipe[1]} คอลัมน์")
print(f"  - ช่วงเวลา: ตั้งแต่ {min_date_pipe.strftime('%Y-%m-%d')} ถึง {max_date_pipe.strftime('%Y-%m-%d')}\n")

print("=== [ผลการเปรียบเทียบและการสูญหายของข้อมูล] ===")
print(f"  - จำนวนวันทำการที่สูญหายหลังจากการ Merge: {num_lost_days} วัน")
print(f"  - คิดเป็นอัตราการสูญหาย: {pct_lost_days:.2f}%")
if num_lost_days > 0:
    print(f"  - ตัวอย่าง 5 วันทำการของฝั่งหุ้นที่ถูกตัดออกไป:")
    for d in sorted(list(lost_dates))[:5]:
        print(f"    • {d.strftime('%Y-%m-%d')}")

###Feature Engineering จากตัวแปรชุดใหม่ (SET50, SP500, SOX, VIX, DXY และคู่เงิน CNY_THB, EUR_THB, JPY_THB, USD_THB)

###รวมฟีเจอร์ใหม่ (X_features_final) เข้ากับชุดฟีเจอร์เดิมของโมเดล

In [ ]:
import numpy as np
import pandas as pd


def ensure_date_column(df, name="df"):
  d = df.copy()
  if "Date" in d.columns:
    d["Date"] = pd.to_datetime(d["Date"], errors="coerce")
    return d.dropna(subset=["Date"])
  if isinstance(d.index, pd.DatetimeIndex) or d.index.name in (
      "Date",
      "date",
      "วันเดือนปี",
  ):
    d = d.reset_index()
    idx_col = d.columns[0]
    d = d.rename(columns={idx_col: "Date"})
    d["Date"] = pd.to_datetime(d["Date"], errors="coerce")
    return d.dropna(subset=["Date"])
  for cand in ["date", "วันเดือนปี", "Date_Parsed", "observation_date"]:
    if cand in d.columns:
      d = d.rename(columns={cand: "Date"})
      d["Date"] = pd.to_datetime(d["Date"], errors="coerce")
      return d.dropna(subset=["Date"])
  raise KeyError(
      f"ไม่พบคอลัมน์หรือ Index วันที่ใน {name} (คอลัมน์ที่มี: {list(d.columns)})"
  )


def calculate_rsi(series, period=14):
  delta = series.diff()
  gain = delta.clip(lower=0)
  loss = (-delta).clip(lower=0)

  avg_gain = gain.rolling(window=period, min_periods=period).mean()
  avg_loss = loss.rolling(window=period, min_periods=period).mean()

  start_idx = avg_gain.first_valid_index()
  if start_idx is not None:
    pos = series.index.get_loc(start_idx)
    g_vals = avg_gain.to_numpy(dtype=float, copy=True)
    l_vals = avg_loss.to_numpy(dtype=float, copy=True)
    gain_arr = gain.to_numpy(dtype=float)
    loss_arr = loss.to_numpy(dtype=float)
    for i in range(pos + 1, len(series)):
      g_vals[i] = (g_vals[i - 1] * (period - 1) + gain_arr[i]) / period
      l_vals[i] = (l_vals[i - 1] * (period - 1) + loss_arr[i]) / period
    avg_gain = pd.Series(g_vals, index=series.index)
    avg_loss = pd.Series(l_vals, index=series.index)

  rs = avg_gain / avg_loss.replace(0, np.nan)
  return 100 - (100 / (1 + rs))


# 1. เตรียมฐานข้อมูลหลักและเรียงลำดับเวลา
df_fe = ensure_date_column(df_daily_analysis, "df_daily_analysis").sort_values(
    "Date"
)
df_fe = df_fe.drop_duplicates(subset=["Date"]).reset_index(drop=True)

index_close_mapping = {
    "SET50": df_new_SET50,
    "SP500": df_new_SP500,
    "SOX": df_new_SOX,
    "VIX": df_new_VIX,
    "DXY": df_new_DXY,
    "CNY_THB": df_new_fx_CNY_THB,
    "JPY_THB": df_new_fx_JPY_THB,
    "USD_THB": df_new_fx_USD_THB,
}

for name, raw_df in index_close_mapping.items():
  if raw_df is None:
    print(f"⚠ ข้าม {name} เพราะไม่พบข้อมูลในหน่วยความจำ")
    continue
  raw_clean = ensure_date_column(raw_df, name)
  close_col = (
      "Close"
      if "Close" in raw_clean.columns
      else ("ล่าสุด" if "ล่าสุด" in raw_clean.columns else None)
  )
  if close_col is None:
    print(f"⚠ ข้าม {name} เพราะไม่พบคอลัมน์ Close / ล่าสุด")
    continue
  raw_clean[f"{name}_Close"] = pd.to_numeric(
      raw_clean[close_col].astype(str).str.replace(",", "", regex=False),
      errors="coerce",
  )
  raw_clean = (
      raw_clean[["Date", f"{name}_Close"]]
      .dropna(subset=[f"{name}_Close"])
      .drop_duplicates(subset=["Date"])
      .sort_values("Date")
  )
  df_fe = pd.merge(df_fe, raw_clean, on="Date", how="left")

df_fe = df_fe.sort_values("Date").reset_index(drop=True)
close_cols = [c for c in df_fe.columns if c.endswith("_Close")]
df_fe[close_cols] = df_fe[close_cols].ffill().bfill()

# 2. คำนวณฟีเจอร์ใหม่ (เฉพาะข้อมูล ณ เวลา <= t)
X_new_features = pd.DataFrame({"Date": df_fe["Date"]})

for asset in ["SET50", "SOX", "SP500"]:
  col = f"{asset}_Close"
  if col in df_fe.columns:
    X_new_features[f"{asset}_Return_3d"] = df_fe[col].pct_change(3)
    X_new_features[f"{asset}_Return_5d"] = df_fe[col].pct_change(5)

if "SET50_Close" in df_fe.columns:
  set50_close = df_fe["SET50_Close"]
  sma20 = set50_close.rolling(window=20).mean()
  X_new_features["SET50_Distance_SMA20"] = (set50_close - sma20) / sma20
  X_new_features["SET50_RSI_14"] = calculate_rsi(set50_close, period=14)
  X_new_features["SET50_Volatility_Roll5"] = set50_close.rolling(window=5).std()

lag_assets = [
    "SP500_Close",
    "SOX_Close",
    "VIX_Close",
    "DXY_Close",
    "USD_THB_Close",
    "CNY_THB_Close",
    "JPY_THB_Close",
]
for asset in lag_assets:
  if asset in df_fe.columns:
    clean_name = asset.replace("_Close", "")
    X_new_features[f"{clean_name}_Lag1"] = df_fe[asset].shift(1)
    X_new_features[f"{clean_name}_Lag2"] = df_fe[asset].shift(2)

# 3. ล้างค่า inf / NaN และจัดแนวกับ Target
X_new_features = X_new_features.replace([np.inf, -np.inf], np.nan)
valid_rows = X_new_features.dropna().index

X_features_final = X_new_features.loc[valid_rows].reset_index(drop=True)
avail_targets = [
    c
    for c in [
        "Date",
        "Import_Avg",
        "Export_Avg",
        "BH_Pct",
        "BDMS_Pct",
        "EGCO_Pct",
        "BCP_Pct",
    ]
    if c in df_fe.columns
]
Y_targets_final = df_fe.loc[valid_rows, avail_targets].reset_index(drop=True)

print(
    f"✓ สร้างฟีเจอร์ใหม่สำเร็จ: {X_features_final.shape[1] - 1} คอลัมน์ | "
    f"แถวที่พร้อมใช้: {len(X_features_final)} แถว"
)

###รวมฟีเจอร์ใหม่ `X_features_final` เข้ากับชุดฟีเจอร์

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. ตรวจสอบข้อมูลตั้งต้น
print("=== ตรวจสอบข้อมูลดิบก่อนการรวม (Pre-Merge Checking) ===")
print(f"ขนาด final_pipeline_df: {final_pipeline_df.shape}")

final_pipeline_df['Date'] = pd.to_datetime(final_pipeline_df['Date'])

# 2. ดึงฟีเจอร์ตั้งต้นจาก final_pipeline_df
original_features_cols = ['Date', 'Brent', 'US_10Y', 'USD_THB', 'Volatility_to_Open', 'Yield_Curve_Slope']
df_orig_feats = final_pipeline_df[original_features_cols].copy()

# ถ้ายืนยันว่ามี X_features_final ที่ไม่ว่างและมีคอลัมน์ 'Date' ค่อยรวมเพิ่ม
if 'X_features_final' in globals() and not X_features_final.empty and 'Date' in X_features_final.columns:
    X_features_final['Date'] = pd.to_datetime(X_features_final['Date'])
    df_merged_features = pd.merge(df_orig_feats, X_features_final, on='Date', how='inner')
else:
    print("ℹ ใช้ฟีเจอร์จาก final_pipeline_df เป็นหลัก (ข้าม X_features_final)")
    df_merged_features = df_orig_feats

# 3. จัดแนวเป้าหมายสำหรับการทำนายล่วงหน้า 5 วัน (5-Day Horizon Target Selection)
original_targets_cols = ['Import_Avg', 'Export_Avg', 'BH_Pct', 'BDMS_Pct', 'EGCO_Pct', 'BCP_Pct']
df_orig_targets = final_pipeline_df[['Date'] + original_targets_cols].copy()

df_targets_5d = df_orig_targets[original_targets_cols].rolling(window=5).sum().shift(-5)
df_targets_5d['Date'] = df_orig_targets['Date']

# 4. รวมฟีเจอร์ (X) กับเป้าหมาย 5 วันข้างหน้า (Y)
df_final_dataset = pd.merge(df_merged_features, df_targets_5d, on='Date', how='inner')
df_final_dataset = df_final_dataset.sort_values(by='Date').reset_index(drop=True)
df_final_dataset = df_final_dataset.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

if df_final_dataset.empty:
    raise ValueError("df_final_dataset ว่างเปล่าหลัง merge หรือ dropna กรุณาตรวจสอบข้อมูลต้นฉบับ")

# 5. แยก X และ Y ออกจากกัน
exclude_from_x = ['Date'] + original_targets_cols
x_features_list = [col for col in df_final_dataset.columns if col not in exclude_from_x]

X_model = df_final_dataset[x_features_list].copy()
Y_model_raw = df_final_dataset[original_targets_cols].copy()
dates_model = df_final_dataset['Date'].copy()

# แปลง Target เป็น Binary Classification (1: ขึ้น (> 0), 0: ลง/คงที่ (<= 0))
Y_model_class = Y_model_raw.apply(lambda x: np.where(x > 0, 1, 0))

print("\n=== [สรุปโครงสร้างข้อมูลสุดท้ายสำหรับขั้นตอนถัดไป] ===")
print(f"จำนวนวันทำงานทั้งหมด (Rows): {df_final_dataset.shape[0]} วัน")
print(f"จำนวนฟีเจอร์ฝั่ง X ทั้งสิ้น: {X_model.shape[1]} คอลัมน์")
print(f"ช่วงข้อมูลวันที่: {dates_model.min().strftime('%Y-%m-%d')} ถึง {dates_model.max().strftime('%Y-%m-%d')}")
print(f"ขนาดของ X_model: {X_model.shape} | ขนาดของ Y_model_class: {Y_model_class.shape}")

print("\nรายชื่อฟีเจอร์ทั้งหมดที่พร้อมนำไปเทรนโมเดล:")
print(list(X_model.columns))

display(df_final_dataset.head(5))

# บันทึก Analytical Dataset ที่สมบูรณ์พร้อมเทรนลงใน data/03_processed/
proc_dir = Path('../data/03_processed')
if not proc_dir.parent.exists():
    proc_dir = Path('data/03_processed')
proc_dir.mkdir(parents=True, exist_ok=True)
final_out_path = proc_dir / 'final_pipeline_data.csv'
df_final_dataset.to_csv(final_out_path, index=False)
print(f"\n✓ บันทึก Final Analytical Dataset สำเร็จที่: {final_out_path.resolve()}")
print(f"มิติข้อมูล: {df_final_dataset.shape[0]} วัน x {df_final_dataset.shape[1]} ฟีเจอร์")


In [ ]:
# ==============================================================================
# เซลล์เพิ่มเติม: Cross-Correlation Heatmap (สลับแกน: หุ้นแนวตั้ง vs ฟีเจอร์แนวนอน)
# ==============================================================================
import os
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. ตั้งค่าฟอนต์ภาษาไทยสำหรับการแสดงผล
font_candidates = [
    Path('./Sarabun-Regular.ttf'),
    Path('../notebooks/Sarabun-Regular.ttf'),
    Path('notebooks/Sarabun-Regular.ttf')
]
font_path = next((p for p in font_candidates if p.exists()), None)
thai_font = fm.FontProperties(fname=str(font_path)) if font_path else None
if font_path:
    plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white")

# 2. โหลด final_pipeline_data.csv อัตโนมัติ
data_candidates = [
    Path('../data/03_processed/final_pipeline_data.csv'),
    Path('data/03_processed/final_pipeline_data.csv'),
    Path('./final_pipeline_data.csv')
]
final_csv = next((p for p in data_candidates if p.exists()), None)
if final_csv is None:
    raise FileNotFoundError("⚠ ไม่พบไฟล์ final_pipeline_data.csv กรุณาตรวจสอบโฟลเดอร์ data/03_processed/")

df_final = pd.read_csv(final_csv)
df_final['Date'] = pd.to_datetime(df_final['Date'])

# 3. กำหนดรายชื่อหุ้นและโฟลเดอร์ intermediate
import_stocks = ['PTTGC', 'TOP', 'IRPC', 'IVL', 'GPSC', 'EGCO', 'RATCH', 'BCP']
export_stocks = ['DELTA', 'HANA', 'KCE', 'CPF', 'TU', 'STA', 'BDMS', 'BH']

inter_candidates = [
    Path('../data/02_intermediate'),
    Path('data/02_intermediate'),
    Path('./data/02_intermediate')
]
inter_path = next((p for p in inter_candidates if p.exists()), None)

# โหลดผลตอบแทนรายวันของหุ้นทั้ง 16 ตัว
df_stocks_all = pd.DataFrame({'Date': df_final['Date']})

if inter_path:
    for stk in import_stocks + export_stocks:
        csv_file = inter_path / f'df_{stk}.csv'
        if csv_file.exists():
            df_temp = pd.read_csv(csv_file)
            df_temp['Date'] = pd.to_datetime(df_temp['Date'])
            if 'Pct_Change' in df_temp.columns:
                if df_temp['Pct_Change'].dtype == 'object':
                    df_temp[stk] = df_temp['Pct_Change'].astype(str).str.rstrip('%').str.replace(',', '').astype(float) / 100.0
                else:
                    df_temp[stk] = pd.to_numeric(df_temp['Pct_Change'], errors='coerce')
            else:
                df_temp[stk] = pd.to_numeric(df_temp['Close'], errors='coerce').pct_change()
            df_stocks_all = pd.merge(df_stocks_all, df_temp[['Date', stk]], on='Date', how='left')

df_stocks_all = df_stocks_all.ffill().bfill()

# 4. ดึง Import_Avg และ Export_Avg
df_stocks_all['Import_Avg'] = df_final['Import_Avg'].values
df_stocks_all['Export_Avg'] = df_final['Export_Avg'].values

ordered_stock_cols = ['Import_Avg'] + import_stocks + ['Export_Avg'] + export_stocks

# 5. แยก 28 ฟีเจอร์ออกจาก Target
target_cols = ['Date', 'Import_Avg', 'Export_Avg', 'BH_Pct', 'BDMS_Pct', 'EGCO_Pct', 'BCP_Pct']
feature_cols = [c for c in df_final.columns if c not in target_cols]

# 6. คำนวณ Correlation Matrix แบบสลับแกน (แถว = หุ้น 18 ตัว, คอลัมน์ = 28 ฟีเจอร์)
df_features = df_final[feature_cols].reset_index(drop=True)
df_targets_all = df_stocks_all[ordered_stock_cols].reset_index(drop=True)

df_cross_corr = pd.concat([df_features, df_targets_all], axis=1)
# สลับให้แถวเป็นหุ้น และคอลัมน์เป็นฟีเจอร์
cross_corr_matrix = df_cross_corr.corr().loc[ordered_stock_cols, feature_cols]

# 7. วาดกราฟ Heatmap แนวนอนกว้างพิเศษ (22 x 10 นิ้ว)
plt.figure(figsize=(22, 10))
sns.heatmap(
    cross_corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-0.4,
    vmax=0.4,
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title("Cross-Correlation Heatmap: หุ้นนำเข้า-ส่งออก (18 ตัว) vs ฟีเจอร์ที่ผ่านการ Engineered (28 ตัว)", 
          fontproperties=thai_font, fontsize=16, fontweight='bold', pad=20)
plt.xticks(fontproperties=thai_font, fontsize=10, rotation=60, ha='right')
plt.yticks(fontproperties=thai_font, fontsize=11, rotation=0)
plt.tight_layout()
plt.show()

# 8. บันทึกตารางลงใน outputs/tables/
out_candidates = [Path('../outputs/tables'), Path('outputs/tables')]
out_dir = next((p for p in out_candidates if p.parent.exists()), Path('outputs/tables'))
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'cross_correlation_stocks_vs_features.csv'
cross_corr_matrix.to_csv(out_csv)
print(f"✓ บันทึกตาราง Cross-Correlation สำเร็จที่: {out_csv.resolve()}")